<a href="https://colab.research.google.com/github/quiquefluque/Python-Finace/blob/main/TASK_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from pandas.core.indexes.accessors import CombinedDatetimelikeProperties
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

#1.Carga y limpieza de datos

path="/content/drive/MyDrive/datasheet/Task 3 and 4_Loan_Data.csv"
datos=pd.read_csv(path)
df=pd.DataFrame(datos)

#2.columna con buckets

# Definimos los límites de los grupos "cortes" Por ejemplo, de 300 a 400, 400 a 500, etc.
cortes = [300, 400, 500, 600, 700, 800, 850]
# Creamos una nueva columna con el nombre del bucket
df['fico_bucket'] = pd.cut(df['fico_score'], bins=cortes,) #pd.cut(columna, bins=cortes)(lista de numero que definen donde empieza y acaba un intervalo, labels=nombres_grupos(lista con nombre de cada categoria))

#3.Calcular media de probabilidad de default para cada grupo

# Agrupamos por bucket y calculamos la media de la columna 'default'
# Como 'default' es 0 o 1, la media es exactamente la probabilidad (PD)
tabla_pd = df.groupby('fico_bucket')['default'].mean().reset_index()#Crea nueva tabla. df.grupoby(columna de lo que agrupa iguales)[columna de lo que hace media].mean().reset.index()(genera el indice de tabla)
tabla_pd.columns = ['FICO_Bucket', 'PD_Real']
print("--- Probabilidad de Default por cada grupo ---")
print(tabla_pd)

#4.Divido datos

df_modelo = pd.get_dummies(df, columns=['fico_bucket'], drop_first=True)#crea una columna por cada elemento de otra columna, y pone un 1 o 0, dificil de entender, por cada corte(grupo crea una colmna en la nueva tabla df modelo)

X = df_modelo.drop(['customer_id', 'default'], axis=1) # Quitamos el ID y el resultado
y = df_modelo['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creamos el modelo
modelo_log = LogisticRegression(max_iter=1000)

modelo_log.fit(X_train, y_train)

# Hacemos predicciones
predicciones = modelo_log.predict(X_test)
probabilidades = modelo_log.predict_proba(X_test)[:, 1]

# Evaluamos la precisión
print(f"\nPrecisión del modelo (Accuracy): {accuracy_score(y_test, predicciones):.2f}")
print(f"Puntuación ROC-AUC: {roc_auc_score(y_test, probabilidades):.2f}")


/tmp/ipykernel_12599/831156808.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tabla_pd = df.groupby('fico_bucket')['default'].mean().reset_index()#Crea nueva tabla. df.grupoby(columna de lo que agrupa iguales)[columna de lo que hace media].mean().reset.index()(genera el indice de tabla)


--- Probabilidad de Default por cada grupo ---
  FICO_Bucket   PD_Real
0  (300, 400]       NaN
1  (400, 500]  0.722581
2  (500, 600]  0.340324
3  (600, 700]  0.139013
4  (700, 800]  0.046111
5  (800, 850]  0.032258

Precisión del modelo (Accuracy): 1.00
Puntuación ROC-AUC: 1.00
